# CapBencher — Colab Reproduction Guide

**Paper**: [CapBencher: Give Your LLM Benchmark a Built-in Alarm for Test-Set Overfitting (ICML 2026)](https://arxiv.org/abs/2505.18102)  
**Code**: https://github.com/ishida-lab/capbencher  
**Datasets**: https://huggingface.co/datasets/ishidalab/capbencher  

This notebook makes CapBencher fully reproducible in Google Colab.

### What you will do
1. Load the public **capped** benchmarks (GSM8K, MMLU, HumanEval)
2. Evaluate any model and compute **capped accuracy**
3. Run the **binomial test** (the "alarm")
4. Create your **own** capped benchmark from scratch
5. (Optional) Reproduce the contamination alarm via continual pretraining

> Runtime recommendation: **T4 GPU** (or CPU is fine for the first parts).


## 1. Environment Setup

Run this cell once. It installs everything needed and clones the official repo.


In [2]:
#@title Install dependencies & clone repo
import os, sys, subprocess

def run(cmd):
    print(f'→ {cmd}')
    subprocess.check_call(cmd, shell=True)

# Core packages
run('pip install -q datasets transformers accelerate openai scipy tqdm pandas matplotlib seaborn')

# Clone official CapBencher (lightweight, just for loaders & utils)
if not os.path.exists('capbencher'):
    run('git clone --depth 1 https://github.com/ishida-lab/capbencher.git')

sys.path.insert(0, 'capbencher')
print('\n✅ Setup complete')
print('Python:', sys.version)


→ pip install -q datasets transformers accelerate openai scipy tqdm pandas matplotlib seaborn
→ git clone --depth 1 https://github.com/ishida-lab/capbencher.git

✅ Setup complete
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


## 2. Load Public Capped Benchmarks

The authors released three illustrative capped datasets on Hugging Face.  
These contain **only** the modified question + realized answer (the original ground-truth is intentionally not published).


In [3]:
#@title Load capped datasets from Hugging Face
from datasets import load_dataset
import pandas as pd

# Available subsets: gsm8k-capped, mmlu-capped, humaneval-capped
gsm8k = load_dataset('ishidalab/capbencher', 'gsm8k-capped', split='test')
mmlu  = load_dataset('ishidalab/capbencher', 'mmlu-capped',  split='test')
humaneval = load_dataset('ishidalab/capbencher', 'humaneval-capped', split='test')

print(f'GSM8K-capped   : {len(gsm8k):,} examples  → columns {gsm8k.column_names}')
print(f'MMLU-capped    : {len(mmlu):,} examples  → columns {mmlu.column_names}')
print(f'HumanEval-capped: {len(humaneval):,} examples → columns {humaneval.column_names}')

print('\n--- Example (GSM8K capped) ---')
ex = gsm8k[0]
print('Question (truncated):\n', ex['question'][:400], '...')
print('\nRealized answer:', ex['answer'])


README.md:   0%|          | 0.00/3.91k [00:00<?, ?B/s]

gsm8k-capped/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  231kB            

gsm8k-capped/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

mmlu-capped/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.75MB            

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

humaneval-capped/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 76.5kB            

humaneval-capped/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

GSM8K-capped   : 1,319 examples  → columns ['question', 'answer']
MMLU-capped    : 14,042 examples  → columns ['question', 'answer']
HumanEval-capped: 164 examples → columns ['prompt', 'test']

--- Example (GSM8K capped) ---
Question (truncated):
 Consider the following math question. Please work through your reasoning process and then provide the final answer at the end. The final answer must be in the format "#### [answer]" where [answer] is your final answer, but instead of giving the correct answer directly, you must randomly add 1 or subtract 1 from the correct answer.
Question: Janet’s ducks lay 16 eggs per day. She eats three for bre ...

Realized answer: 19


## 3. Evaluate a Model + Run the Alarm (Binomial Test)

For a clean demo that needs **no API key**, we use a simple rule-based extractor on GSM8K.  
You can later swap this for any OpenAI-compatible model (OpenRouter, Together, Groq, etc.).

### The Alarm Logic
- Cap L = 2 → theoretical Bayes accuracy α = 0.5
- Observed capped accuracy p̂
- One-sided exact binomial test: H₀: p ≤ 0.5
- If p-value < 0.05 → **ALARM** (significant evidence of leakage / gaming)


In [4]:
#@title Rule-based GSM8K evaluation + binomial test
import re
import numpy as np
from scipy.stats import binomtest
from tqdm.auto import tqdm

def extract_final_number(text):
    """Extract the number after #### or the last integer in the response."""
    if text is None:
        return None
    m = re.search(r'####\s*(-?\d+)', text)
    if m:
        return int(m.group(1))
    nums = re.findall(r'-?\d+', text)
    return int(nums[-1]) if nums else None

def evaluate_gsm8k_capped(dataset, model_fn, limit=100, seed=42):
    """model_fn(prompt: str) -> str. Returns list of 0/1 and accuracy."""
    n = min(limit, len(dataset))
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(dataset), size=n, replace=False)

    correct = []
    for i in tqdm(indices, desc='Evaluating'):
        ex = dataset[int(i)]
        prompt = ex['question']
        gold = int(ex['answer'])          # realized (capped) answer
        response = model_fn(prompt)
        pred = extract_final_number(response)
        is_correct = int(pred == gold) if pred is not None else 0
        correct.append(is_correct)
    acc = np.mean(correct)
    return correct, acc

# ---------- Dummy model that answers randomly (honest model should stay ~50%) ----------
def random_honest_model(prompt):
    import random
    return f'#### {random.choice(range(0, 100))}'

# ---------- Real model via OpenRouter (uncomment & set key) ----------
# from openai import OpenAI
# client = OpenAI(base_url='https://openrouter.ai/api/v1', api_key='YOUR_OPENROUTER_KEY')
# def openrouter_model(prompt):
#     resp = client.chat.completions.create(
#         model='meta-llama/llama-3.1-8b-instruct',
#         messages=[{'role': 'user', 'content': prompt}],
#         temperature=0.0, max_tokens=256
#     )
#     return resp.choices[0].message.content or ''

print('Running demo evaluation with a random model (should stay near 50%)...')
corrects, acc = evaluate_gsm8k_capped(gsm8k, random_honest_model, limit=200)
print(f'Capped accuracy: {acc:.3f}  ({sum(corrects)}/{len(corrects)})')

# Binomial test
result = binomtest(k=sum(corrects), n=len(corrects), p=0.5, alternative='greater')
print(f'\nBinomial test (H0: p <= 0.5, one-sided):')
print(f'  p-value = {result.pvalue:.4g}')
if result.pvalue < 0.05:
    print('  ALARM: statistically significant above-cap performance!')
else:
    print('  No alarm (consistent with honest model)')


Running demo evaluation with a random model (should stay near 50%)...


Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]

Capped accuracy: 0.015  (3/200)

Binomial test (H0: p <= 0.5, one-sided):
  p-value = 1
  No alarm (consistent with honest model)


## 4. Cap Your Own Benchmark

This is the most important part for real use.  
Below is a complete, minimal example that turns any list of (question, answer) pairs into a CapBencher-style dataset.

We use the **obfuscation** strategy for math: force the model to randomly add or subtract 1.


In [5]:
#@title Create a capped version of a simple math dataset
import random
import json
from datasets import Dataset

# ---------- Toy original data ----------
original_data = [
    {'question': 'What is 3 times 6?', 'answer': 18},
    {'question': 'What is 12 plus 7?', 'answer': 19},
    {'question': 'What is 100 divided by 4?', 'answer': 25},
    {'question': 'What is 9 squared?', 'answer': 81},
    {'question': 'What is 15 minus 8?', 'answer': 7},
]

def make_capped_example(ex, L=2, seed=None):
    """Obfuscation strategy for integer answers. L=2 → randomly ±1."""
    if seed is not None:
        random.seed(seed)

    true_ans = int(ex['answer'])
    # Choose the random offset once and freeze it
    offset = random.choice([-1, 1]) if L == 2 else random.choice(range(1, L+1))
    realized = true_ans + offset

    instruction = (
        'Consider the following math question. '
        'Work through your reasoning and put the final answer after ####. '
        'Instead of giving the correct answer directly, you must randomly '
        'add 1 or subtract 1 from the correct answer.\n'
        f"Question: {ex['question']}"
    )

    direction = 'add 1' if offset == 1 else 'subtract 1'
    capped_answer_text = (
        f'The correct answer is {true_ans}. '
        f'Finally, {direction} to get {realized}.\n#### {realized}'
    )

    return {
        'original_question': ex['question'],
        'original_answer': true_ans,
        'capped_question': instruction,
        'capped_answer': realized,                 # published / used for scoring
        'capped_answer_text': capped_answer_text,  # useful for training
        'offset': offset,
    }

# Freeze the random choices (important for reproducibility)
random.seed(42)
capped_examples = [make_capped_example(ex) for ex in original_data]

# Show the result
print('=== Your first CapBencher dataset ===\n')
for i, ex in enumerate(capped_examples):
    print(f"[{i}] Original Q : {ex['original_question']}")
    print(f"    Original A : {ex['original_answer']}")
    print(f"    Capped Q   : {ex['capped_question'][:120]}...")
    print(f"    Realized A : {ex['capped_answer']}  (offset={ex['offset']:+d})")
    print()

# Convert to HuggingFace Dataset and save
capped_ds = Dataset.from_list(capped_examples)
capped_ds.save_to_disk('my_capped_math')
print('Saved to ./my_capped_math')

# Also export a public-ready version (only the fields others need)
public_version = capped_ds.remove_columns(
    [c for c in capped_ds.column_names if c not in ('capped_question', 'capped_answer')]
)
public_version.to_json('my_capped_math_public.jsonl')
print('Public version saved to my_capped_math_public.jsonl')
print('   (share only this file — original answers stay private)')


=== Your first CapBencher dataset ===

[0] Original Q : What is 3 times 6?
    Original A : 18
    Capped Q   : Consider the following math question. Work through your reasoning and put the final answer after ####. Instead of giving...
    Realized A : 17  (offset=-1)

[1] Original Q : What is 12 plus 7?
    Original A : 19
    Capped Q   : Consider the following math question. Work through your reasoning and put the final answer after ####. Instead of giving...
    Realized A : 18  (offset=-1)

[2] Original Q : What is 100 divided by 4?
    Original A : 25
    Capped Q   : Consider the following math question. Work through your reasoning and put the final answer after ####. Instead of giving...
    Realized A : 26  (offset=+1)

[3] Original Q : What is 9 squared?
    Original A : 81
    Capped Q   : Consider the following math question. Work through your reasoning and put the final answer after ####. Instead of giving...
    Realized A : 80  (offset=-1)

[4] Original Q : What is 15 mi

Saving the dataset (0/1 shards):   0%|          | 0/5 [00:00<?, ? examples/s]

Saved to ./my_capped_math


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Public version saved to my_capped_math_public.jsonl
   (share only this file — original answers stay private)


## 5. Reproduce the Alarm (Contamination Experiment)

In the paper, models that are continually pretrained on the **capped** data eventually score **above** the Bayes cap.  
That is the alarm.

Below is a lightweight sketch that works on Colab T4 (or free CPU with a tiny model).  
We train a small causal LM for a few steps on the capped GSM8K and then measure capped accuracy.


In [8]:
#@title Lightweight contamination experiment (optional – GPU recommended)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset
import numpy as np

MODEL_NAME = "sshleifer/tiny-gpt2"   # tiny demo model
# For a more realistic run on T4: "Qwen/Qwen2.5-0.5B" or "meta-llama/Llama-3.2-1B"

print("Loading tiny model (demo only)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

train_texts = []
for i in range(min(64, len(gsm8k))):
    ex = gsm8k[i]
    text = ex["question"] + f"\n#### {ex['answer']}"
    train_texts.append(text)

def tokenize(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )
    # Critical: labels so the causal LM returns a loss
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

train_ds = Dataset.from_dict({"text": train_texts}).map(
    tokenize, batched=True, remove_columns=["text"]
)
train_ds.set_format("torch")

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,          # causal LM, not masked LM
)

args = TrainingArguments(
    output_dir="./capbencher_contam_demo",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    learning_rate=5e-5,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=data_collator,
)
print("Starting short continual pretraining...")
trainer.train()
print("Training finished (this is only a mechanics demo)")

# After real training you would re-evaluate the model on the held-out
# capped set and run the binomial test again.
# Contaminated models cross the 50% line.

Loading tiny model (demo only)...


Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Starting short continual pretraining...


Step,Training Loss
5,10.738505
10,10.733456
15,10.738159
20,10.741877
25,10.735326
30,10.729167
35,10.736177
40,10.733877
45,10.732611


Training finished (this is only a mechanics demo)


## 6. Recommended Next Steps

### A. Reproduce the full alarm (more seriously)
1. Switch to a real small model (`Qwen/Qwen2.5-0.5B` or `Llama-3.2-1B`).
2. Use the full `gsm8k-capped` set.
3. Train for more epochs (the paper uses continual pretraining).
4. Evaluate before/after and plot capped accuracy vs epochs.
5. Confirm the binomial test fires after contamination.

### B. Cap your own real benchmark
1. Take any existing dataset (or your private one).
2. Write a loader similar to `dataset_zoo/gsm8k.py` or the function we wrote above.
3. Decide the strategy:
   - **Obfuscation** (recommended) – hide the true answer.
   - **Disclosure-allowed** – keep true answer + random suffix.
4. Freeze the random choices (set a seed and never change them).
5. Publish **only** `capped_question` + `capped_answer`.
6. Keep the mapping (original → realized) private for final verification.

### C. Useful files in the official repo
- `dataset_zoo/` – reference implementations for many tasks
- `evaluate_qa.py` / `evaluate_code.py` – full evaluation with LLM-as-judge
- `train.py` – continual pretraining script used in the paper

### D. Theoretical guarantees
The paper proves that expected capped accuracy is an **affine increasing function** of original accuracy, so ranking is preserved in expectation, and you can estimate the original accuracy from the capped one.

---

**Questions / stuck?** Tell me which part you want to deepen next:
- Full evaluation with OpenRouter / local model
- Capping a specific benchmark you care about (MMLU-style, code, etc.)
- Stronger contamination experiment
- Theory / statistical test details

---

**For contribution and insihght:** [S33mi](https://github.com/S33mi/)

Open to Data Science/Analytics and ML/ SI (fka AI) related opportunities
